In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm

SEED = 692

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

### Dataset

In [ ]:
# Get the data and labels
iris = load_iris()
X, y = iris.data, iris.target

# Split the data into training, validation, and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=SEED)

print(f"Train: data { X_train.shape}, labels {y_train.shape}")
print(f"Validation: data { X_val.shape}, labels {y_val.shape}")
print(f"Test: data { X_test.shape}, labels {y_test.shape}")

print(f"Example of train data: {X_train[0]}, label: {y_train[0]}")
print(f"Example of validation data: {X_val[0]}, label: {y_val[0]}")
print(f"Example of test data: {X_test[0]}, label: {y_test[0]}")

dataset = {}
dataset["train"] = {"data": X_train, "labels": y_train}
dataset["val"] = {"data": X_val, "labels": y_val}
dataset["test"] = {"data": X_test, "labels": y_test}
dataset["class_labels"] = [iris.target_names[0], iris.target_names[1], iris.target_names[2]]

### Architecture definition

In [ ]:
from torchsummary import summary


class ANN(nn.Module):
    def __init__(
        self,
        input_size: int = 4,
        output_size: int = 3,
        output_bias: bool = True,
        layers_dims: list = [16, 16],
    ):
        super().__init__()
        self.input_size = input_size
        self.output_size = output_size
        self.layers_dims = layers_dims

        self.layers = nn.ModuleList()
        layers = [input_size] + layers_dims
        num_layers = len(layers) - 1
        for i in range(num_layers):
            in_features = layers[i]
            out_features = layers[i + 1]
            self.layers.append(
                nn.Sequential(
                    nn.Linear(in_features, out_features),
                    nn.ReLU(),
                )
            )
        
        self.output_layer = nn.Linear(layers[-1], output_size, bias=output_bias)

    def forward(self, x: torch.Tensor, debug: bool = False) -> torch.Tensor:
        for layer in self.layers:
            if debug:
                print(f"Input shape: {x.shape}")
            x = layer(x)
        
        if debug:
            print(f"Last input shape: {x.shape}")
        x = self.output_layer(x)

        return x


net = ANN(output_size=3, layers_dims=[10, 20])

a = torch.randn(1, 4)
b = net(a, debug=True)

summary(net.to(device), input_size=(1, 4), batch_size=1)

### Training functions

In [ ]:
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader, TensorDataset
from typing import Literal, Optional, Union


def train(
    net: nn.Module,
    dataset: dict,
    train_mode: Literal["full-dataset", "single-sample", "mini-batch"] = "full-dataset",
    prefix: Optional[str] = None,
    checkpoints_path: Union[str, Path] = "./checkpoints",
    upper_bound: float = 1.0,
    save_model: bool = False,
    epochs: int = 100,
    lr: float = 1e-1,
    device: torch.device = device,
    layers2tensorboard: bool = True,
    lambda_reg: float = 0.0
):
    tensorboard_path = "./tensorboard"

    # Create checkpoints directory if it doesn't exist
    checkpoints_path = Path(checkpoints_path)
    checkpoints_path.mkdir(parents=True, exist_ok=True)

    net.to(device)

    optimizer = optim.SGD(net.parameters(), lr=lr, weight_decay=lambda_reg)
    criterion = nn.CrossEntropyLoss()

    now = datetime.now()
    suffix = now.strftime("%Y-%m-%d_%H-%M-%S")
    prefix = suffix if prefix is None else f"{prefix}_{suffix}"
    writer = SummaryWriter(log_dir=f"{tensorboard_path}/{prefix}")

    accs = {"train": [], "val": []}
    max_acc = 0.0
    best_model = None

    if train_mode == "mini-batch":
        batch_size = 16
        train_x = torch.from_numpy(dataset["train"]["data"]).float().to(device)
        train_y = torch.from_numpy(dataset["train"]["labels"]).long().to(device)
        train_dataset = TensorDataset(train_x, train_y)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    elif train_mode == "single-sample":
        batch_size = 1
        train_x = torch.from_numpy(dataset["train"]["data"]).float().to(device)
        train_y = torch.from_numpy(dataset["train"]["labels"]).long().to(device)
        train_dataset = TensorDataset(train_x, train_y)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    elif train_mode == "full-dataset":
        train_x = torch.from_numpy(dataset["train"]["data"]).float().to(device)
        train_y = torch.from_numpy(dataset["train"]["labels"]).long().to(device)
        train_loader = [(train_x, train_y)]
    else:
        raise ValueError(f"Invalid train_mode: {train_mode}")

    writer.add_graph(net, train_x)

    pb = tqdm(range(epochs), desc=f"Training using {train_mode} mode", unit="epoch")
    for epoch in pb:
        net.train()
        optimizer.zero_grad()

        # Forward pass, here we simulate different training modes by using the appropriate DataLoader
        batch_accs = []
        for batch_x, batch_y in train_loader:
            y_hat = net(batch_x)

            # Compute loss
            error = criterion(y_hat, batch_y)

            # Backward pass and optimization
            error.backward()
            optimizer.step()

            # Compute training accuracy
            _, predicted = torch.max(y_hat, 1)
            acc_train = (predicted == batch_y).float().mean().item()
            batch_accs.append(acc_train)

        acc_train = np.mean(batch_accs)
        # Compute validation accuracy
        acc_val, error_val, cf = validate(net, dataset, criterion, device)

        if layers2tensorboard:
            plot_layers_to_tensorboard(net, writer, epoch)

        if acc_val > max_acc:
            max_acc = acc_val
            best_model = net.state_dict()

        # Update the progress bar with the current metrics
        pb.set_postfix({
            "train_loss": error.item(),
            "train_acc": acc_train,
            "val_loss": error_val,
            "val_acc": acc_val,
            "max_val_acc": max_acc,
        })

        accs["train"].append(acc_train)
        accs["val"].append(acc_val)

        writer.add_scalar("Loss/train", error.item(), epoch)
        writer.add_scalar("Accuracy/train", acc_train, epoch)
        writer.add_scalar("Loss/val", error_val, epoch)
        writer.add_scalar("Accuracy/val", acc_val, epoch)
        writer.add_figure("Confusion Matrix/val", ConfusionMatrixDisplay(cf, display_labels=dataset["class_labels"]).plot().figure_, epoch)

        if acc_val >= upper_bound:
            print(f"Early stopping at epoch {epoch} with validation accuracy {acc_val:.4f}")
            break
    
    if save_model and best_model is not None:
        torch.save(best_model, f"{checkpoints_path}/{prefix}-{epoch}-{max_acc:.4f}-{train_mode}.pth")

    first_full_acc = next((i for i, acc in enumerate(accs["val"]) if acc >= upper_bound), None)
    # Register number of epochs to reach upper_bound in TensorBoard with the mode as a tag
    if first_full_acc is not None:
        writer.add_scalar(f"Epochs to reach {upper_bound:.4f}", first_full_acc, 0)

    writer.flush()
    writer.close()

    return accs, best_model, first_full_acc



def validate(
    net: nn.Module,
    dataset: dict,
    criterion: nn.Module,
    device: torch.device = device,
) -> tuple[float, float, np.ndarray]:
    val_x = torch.from_numpy(dataset["val"]["data"]).float().to(device)
    val_y = torch.from_numpy(dataset["val"]["labels"]).long().to(device)

    net.to(device)
    net.eval()
    
    with torch.no_grad():
        y_hat = net(val_x)
        error = criterion(y_hat, val_y)

        _, predicted = torch.max(y_hat, 1)
        acc_val = (predicted == val_y).float().mean().item()

    cf = confusion_matrix(val_y.cpu(), predicted.cpu())

    return acc_val, error.item(), cf


def test(
    net: nn.Module,
    dataset: dict,
    device: torch.device = device,
) -> tuple[float, np.ndarray]:
    test_x = torch.from_numpy(dataset["test"]["data"]).float().to(device)
    test_y = torch.from_numpy(dataset["test"]["labels"]).long().to(device)

    net.to(device)
    net.eval()
    
    with torch.no_grad():
        y_hat = net(test_x)
        _, predicted = torch.max(y_hat, 1)
        acc_test = (predicted == test_y).float().mean().item()

    cf = confusion_matrix(test_y.cpu(), predicted.cpu())

    return acc_test, cf


def plot_layers_to_tensorboard(net: nn.Module, writer: SummaryWriter, epoch: int):
    for name, param in net.named_parameters():
        if any(keyword in name for keyword in ["weight", "bias"]):
            writer.add_histogram(name, param.cpu().data.numpy(), epoch)

### Run training phase

In [ ]:
import pandas as pd

epochs = 1000
lr = 2e-2
lambda_reg = 1e-4
prefix = f"ANN-Iris-e-{epochs}-lr-{lr}-lambda_reg-{lambda_reg}"
checkpoints_path = "./checkpoints"
layers = [8, 16]


# Train the model using different training modes and compare their performance
accs_dict = {}
first_full_accs = {}
for mode in ["full-dataset", "single-sample", "mini-batch"]:
    # Re-initialize the model for each training mode to ensure a fair comparison
    net = ANN(output_size=3, layers_dims=layers)

    accs, best_model, first_full_acc = train(
        net=net,
        dataset=dataset,
        train_mode=mode,
        prefix=prefix,
        checkpoints_path=checkpoints_path,
        save_model=True,
        epochs=epochs,
        lr=lr,
        lambda_reg=lambda_reg,
        device=device,
    )
    accs_dict[mode] = accs
    first_full_accs[mode] = first_full_acc if first_full_acc is not None else epochs

pd.DataFrame(first_full_accs, index=["Epochs taken to reach 100% accuracy"]).T

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 3, figsize=(18, 5))
for mode in accs_dict.keys():
    key_idx = list(accs_dict.keys()).index(mode)
    axs[key_idx].plot(accs_dict[mode]["train"], label="Train Accuracy")
    axs[key_idx].plot(accs_dict[mode]["val"], label="Validation Accuracy")
    axs[key_idx].set_title(f"{mode} - Train vs Validation Accuracy")
    axs[key_idx].set_xlabel("Epoch")
    axs[key_idx].set_ylabel("Accuracy")
    axs[key_idx].legend()
    axs[key_idx].grid()
fig.suptitle("Training and Validation Accuracy for Different Training Modes", fontsize=16)
fig.tight_layout()
fig.show()

### Run testing phase

In [ ]:
full_dataset_net_path = next(Path(checkpoints_path).glob("*-full-dataset.pth"))
single_sample_net_path = next(Path(checkpoints_path).glob("*-single-sample.pth"))
mini_batch_net_path = next(Path(checkpoints_path).glob("*-mini-batch.pth"))

fig, axs = plt.subplots(1, 3, figsize=(18, 5))
for mode, net_path in zip(["full-dataset", "single-sample", "mini-batch"], [full_dataset_net_path, single_sample_net_path, mini_batch_net_path]):
    net = ANN(output_size=3, layers_dims=layers)
    net.load_state_dict(torch.load(net_path))
    acc_val, cf = test(net.to(device), dataset, device)
    key_idx = list(["full-dataset", "single-sample", "mini-batch"]).index(mode)
    axs[key_idx].set_title(f"{mode} - Val Acc: {acc_val:.4f}")
    ConfusionMatrixDisplay(cf, display_labels=dataset["class_labels"]).plot(ax=axs[key_idx])
plt.show()

### Run single inference

In [ ]:
def run_single_inference(net: nn.Module, dataset: dict, idx: int, device: torch.device = device):
    test_x = torch.from_numpy(dataset["test"]["data"]).float().to(device)
    test_y = torch.from_numpy(dataset["test"]["labels"]).long().to(device)

    sample_x = test_x[idx].unsqueeze(0)
    sample_y = test_y[idx].unsqueeze(0)

    net.eval()
    net.to(device)
    with torch.no_grad():
        y_hat = net(sample_x)

    _, predicted = torch.max(y_hat, 1)
    true_label = dataset["class_labels"][sample_y.item()]
    predicted_label = dataset["class_labels"][predicted.item()]

    target_features = sample_x.cpu().numpy().flatten()
    predicted_label_example_features = test_x[test_y == predicted.item()].mean(0).cpu().numpy()

    return target_features, predicted_label_example_features, true_label, predicted_label


idx = np.random.randint(0, len(dataset["test"]["data"]))
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
for mode, net_path in zip(["full-dataset", "single-sample", "mini-batch"], [full_dataset_net_path, single_sample_net_path, mini_batch_net_path]):
    net = ANN(output_size=3, layers_dims=layers)
    net.load_state_dict(torch.load(net_path))
    target_features, predicted_label_example_features, true_label, predicted_label = run_single_inference(net, dataset, idx=idx, device=device)

    hit_or_miss = "HIT" if true_label == predicted_label else "MISS"
    # set title color to green if hit, red if miss
    axs[list(["full-dataset", "single-sample", "mini-batch"]).index(mode)].set_title(
        f"({hit_or_miss}) {mode} - True: {true_label}, Predicted: {predicted_label}",
        color="green" if hit_or_miss == "HIT" else "red"
    )
    
    x = np.arange(len(target_features))
    key_idx = list(["full-dataset", "single-sample", "mini-batch"]).index(mode)
    axs[key_idx].bar(x - 0.2, target_features, width=0.4, label="Target Sample", color="blue")
    axs[key_idx].bar(x + 0.2, predicted_label_example_features, width=0.4, label="Predicted Label Avg", color="orange")
    axs[key_idx].set_xticks(x)
    axs[key_idx].set_xticklabels(iris.feature_names, rotation=45)
    axs[key_idx].legend()
plt.tight_layout()
plt.show()

### Visualize Tensorboard

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir ./tensorboard --port 6006